In [67]:
from pathlib import Path

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent


In [68]:
# %% 미추출 zip 압축 해제
import zipfile

RAW_DIR = project_root / 'data' / 'raw'
EXTRACTED_DIR = project_root / 'data' / 'extracted'

zip_files = sorted(RAW_DIR.glob('*.zip'))
print(f'raw zip 파일 총 {len(zip_files)}개\n')

for zp in zip_files:
    target = EXTRACTED_DIR / zp.stem
    if target.exists():
        print(f'  [SKIP] {zp.name}  →  이미 추출됨')
    else:
        print(f'  [UNZIP] {zp.name}  →  {target.name}/ ...', end='', flush=True)
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(target)
        print(' 완료')

print('\n✅ 압축 해제 완료')


raw zip 파일 총 9개

  [SKIP] 상권분석서비스(점포_상권).zip  →  이미 추출됨
  [SKIP] 서울시 상권분석서비스(길단위인구-상권).zip  →  이미 추출됨
  [SKIP] 서울시 상권분석서비스(상권변화지표-상권).zip  →  이미 추출됨
  [SKIP] 서울시 상권분석서비스(상주인구-상권).zip  →  이미 추출됨
  [SKIP] 서울시 상권분석서비스(직장인구-상권).zip  →  이미 추출됨
  [SKIP] 서울시_내외국인_관광매출액.zip  →  이미 추출됨
  [SKIP] 서울시_상권분석서비스(추정매출+영역).zip  →  이미 추출됨
  [SKIP] 서울시_외부데이터추가.zip  →  이미 추출됨
  [SKIP] 서울시_전월세가.zip  →  이미 추출됨

✅ 압축 해제 완료


In [69]:
# %% 전체 데이터 로드
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

DATA_DIR  = project_root / 'data' / 'extracted' / '서울시_상권분석서비스(추정매출+영역)'
STORE_DIR = project_root / 'data' / 'extracted' / '상권분석서비스(점포_상권)'
FLOAT_DIR = project_root / 'data' / 'extracted' / '서울시 상권분석서비스(길단위인구-상권)'
TOUR_DIR  = project_root / 'data' / 'extracted' / '서울시_내외국인_관광매출액'
RENT_DIR  = project_root / 'data' / 'extracted' / '서울시_전월세가'

sales = pd.concat(
    [pd.read_csv(f, encoding='cp949', low_memory=False) for f in sorted(DATA_DIR.glob('*추정매출*.csv'))],
    ignore_index=True
)
area = pd.read_csv(sorted(DATA_DIR.glob('*영역*상권*.csv'))[0], encoding='cp949')
df_점포 = pd.concat(
    [pd.read_csv(f, encoding='cp949', low_memory=False) for f in sorted(STORE_DIR.glob('*.csv'))],
    ignore_index=True
)
floating = pd.read_csv(
    FLOAT_DIR / '서울시 상권분석서비스(길단위인구-상권).csv', encoding='cp949'
)
df_직장 = pd.read_csv(
    project_root / 'data' / '06_직장인구' / '서울시 상권분석서비스(직장인구-상권).csv', encoding='cp949'
)
df_상주 = pd.read_csv(
    project_root / 'data' / '07_상주인구' / '서울시 상권분석서비스(상주인구-상권).csv', encoding='cp949'
)
df_내국인관광 = pd.read_csv(TOUR_DIR / '서울시_내국인_추정매출액_통합.csv', encoding='utf-8')
df_외국인관광 = pd.read_csv(TOUR_DIR / '서울시_외국인_추정매출액_통합.csv', encoding='utf-8')

# 빈 리스트 생성
df_list = []

# 폴더 내 모든 CSV 파일을 하나씩 읽기
for f in sorted(RENT_DIR.glob('*.csv')):
    try:
        # 1. 먼저 utf-8로 시도
        df = pd.read_csv(f, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            # 2. 실패하면 cp949 (EUC-KR 호환)로 시도
            df = pd.read_csv(f, encoding='cp949')
        except UnicodeDecodeError:
            # 3. 그래도 실패하면 엑셀용 utf-8 (BOM)로 시도
            df = pd.read_csv(f, encoding='utf-8-sig')
            
    df_list.append(df)

# 정상적으로 읽힌 데이터프레임들을 하나로 합치기
df_전월세 = pd.concat(df_list, ignore_index=True)

print(f"총 {len(df_list)}개의 파일이 성공적으로 병합되었습니다.")

# BOM 제거
for df in [df_내국인관광, df_외국인관광]:
    df.columns = df.columns.str.strip().str.replace('\ufeff', '')

print(f'추정매출: {sales.shape}')
print(f'점포:     {df_점포.shape}')
print(f'유동인구: {floating.shape}')
print(f'직장인구: {df_직장.shape}')
print(f'상주인구: {df_상주.shape}')
print(f'관광매출: 내국인 {df_내국인관광.shape} / 외국인 {df_외국인관광.shape}')
print(f'전월세가: {df_전월세.shape}')


총 5개의 파일이 성공적으로 병합되었습니다.
추정매출: (519931, 55)
점포:     (1831925, 14)
유동인구: (46184, 27)
직장인구: (45840, 26)
상주인구: (40812, 29)
관광매출: 내국인 (2989, 9) / 외국인 (2976, 9)
전월세가: (3238096, 23)


In [70]:
# %% 전처리
# ── 연도 추출 ──
for df in [sales, df_점포, floating, df_직장, df_상주]:
    df['기준_년도'] = (df['기준_년분기_코드'] // 10).astype(int)

# ── 자치구 매핑 (자치구_코드_명만 추가, 상권_구분_코드_명은 원본에 있음) ──
gu_map = area[['상권_코드', '자치구_코드_명', '상권_구분_코드_명']].drop_duplicates()

sales    = sales.merge(gu_map[['상권_코드', '자치구_코드_명']], on='상권_코드', how='left')
df_점포  = df_점포.merge(gu_map[['상권_코드', '자치구_코드_명']], on='상권_코드', how='left')
floating = floating.merge(gu_map[['상권_코드', '자치구_코드_명']], on='상권_코드', how='left')
df_직장  = df_직장.merge(gu_map[['상권_코드', '자치구_코드_명']], on='상권_코드', how='left')
df_상주  = df_상주.merge(gu_map[['상권_코드', '자치구_코드_명']], on='상권_코드', how='left')


# ── 관광매출 전처리 (wide → long, 단위: 천원 → 원) ──
def prep_tour(df, label):
    yr_cols = [c for c in df.columns if '매출액' in c]
    melted = df.melt(id_vars=['자치구'], value_vars=yr_cols,
                     var_name='_yr', value_name='관광매출_천원')
    melted['기준_년도'] = melted['_yr'].str.extract(r'(\d{4})').astype(int)
    agg = melted.groupby(['기준_년도', '자치구'])['관광매출_천원'].sum().reset_index()
    agg[f'{label}_관광매출'] = agg['관광매출_천원'] * 1000
    return agg[['기준_년도', '자치구', f'{label}_관광매출']].rename(columns={'자치구': '자치구_코드_명'})

tour_내 = prep_tour(df_내국인관광, '내국인')
tour_외 = prep_tour(df_외국인관광, '외국인')
df_관광 = tour_내.merge(tour_외, on=['기준_년도', '자치구_코드_명'], how='outer').fillna(0)
df_관광['총_관광매출'] = df_관광['내국인_관광매출'] + df_관광['외국인_관광매출']

# ── 전월세가 전처리 (월세 계약 중앙값) ──
df_임대 = (
    df_전월세[df_전월세['전월세구분'] == '월세']
    .rename(columns={'접수년도': '기준_년도', '자치구명': '자치구_코드_명'})
    .groupby(['기준_년도', '자치구_코드_명'])
    .agg(
        중위_월세_임대료 = ('임대료(만원)', 'median'),
        중위_월세_보증금 = ('보증금(만원)', 'median'),
    )
    .reset_index()
)

print('전처리 완료')
print(f'관광매출 연도 범위: {df_관광["기준_년도"].min()} ~ {df_관광["기준_년도"].max()}')
print(f'전월세가 연도 범위: {df_임대["기준_년도"].min()} ~ {df_임대["기준_년도"].max()}')


전처리 완료
관광매출 연도 범위: 2020 ~ 2025
전월세가 연도 범위: 2021 ~ 2025


# 집계 테이블

In [71]:
# %% 자치구별_기초집계
# ── 매출 집계 ──
agg_sales = sales.groupby(['기준_년도', '자치구_코드_명']).agg(
    연간_총매출액   = ('당월_매출_금액',        'sum'),
    연간_총결제건수 = ('당월_매출_건수',         'sum'),
    주중_매출합     = ('주중_매출_금액',          'sum'),
    주말_매출합     = ('주말_매출_금액',          'sum'),
    점심_매출합     = ('시간대_11~14_매출_금액',  'sum'),
    저녁_매출합     = ('시간대_17~21_매출_금액',  'sum'),
    상권_수         = ('상권_코드',              'nunique'),
).reset_index()

agg_sales['평균_객단가']  = agg_sales['연간_총매출액'] / agg_sales['연간_총결제건수'].replace(0, np.nan)
agg_sales['주중_매출비중'] = agg_sales['주중_매출합']  / agg_sales['연간_총매출액']
agg_sales['주말_매출비중'] = agg_sales['주말_매출합']  / agg_sales['연간_총매출액']
agg_sales['점심_매출비중'] = agg_sales['점심_매출합']  / agg_sales['연간_총매출액']
agg_sales['저녁_매출비중'] = agg_sales['저녁_매출합']  / agg_sales['연간_총매출액']
agg_sales = agg_sales.drop(columns=['주중_매출합', '주말_매출합', '점심_매출합', '저녁_매출합'])

# ── 점포 집계 (업종 중복 → 상권+분기 단위 먼저 합산) ──
점포_상권분기 = df_점포.groupby(
    ['기준_년도', '자치구_코드_명', '기준_년분기_코드', '상권_코드']
).agg(
    점포_수          = ('점포_수',          'sum'),
    개업_점포_수     = ('개업_점포_수',      'sum'),
    폐업_점포_수     = ('폐업_점포_수',      'sum'),
    프랜차이즈_점포_수 = ('프랜차이즈_점포_수', 'sum'),
).reset_index()

agg_store = 점포_상권분기.groupby(['기준_년도', '자치구_코드_명']).agg(
    총_점포수      = ('점포_수',           'mean'),  # 시점 재고 → 분기 평균
    총_개업점포수  = ('개업_점포_수',       'sum'),
    총_폐업점포수  = ('폐업_점포_수',       'sum'),
    총_프랜차이즈  = ('프랜차이즈_점포_수',  'mean'),
).reset_index()

agg_store['프랜차이즈_비율'] = agg_store['총_프랜차이즈']  / agg_store['총_점포수'].replace(0, np.nan)
agg_store['개업률']         = agg_store['총_개업점포수']   / agg_store['총_점포수'].replace(0, np.nan)
agg_store['폐업률']         = agg_store['총_폐업점포수']   / agg_store['총_점포수'].replace(0, np.nan)
agg_store['순성장률']       = agg_store['개업률'] - agg_store['폐업률']

# ── 인구 집계 ──
agg_float = floating.groupby(['기준_년도', '자치구_코드_명']).agg(
    총_유동인구 = ('총_유동인구_수', 'sum')
).reset_index()

agg_work = df_직장.groupby(['기준_년도', '자치구_코드_명']).agg(
    총_직장인구 = ('총_직장_인구_수', 'sum')
).reset_index()

agg_res = df_상주.groupby(['기준_년도', '자치구_코드_명']).agg(
    총_상주인구 = ('총_상주인구_수', 'sum')
).reset_index()

# ── 전체 병합 ──
자치구별_기초집계 = (
    agg_sales
    .merge(agg_store,  on=['기준_년도', '자치구_코드_명'], how='left')
    .merge(agg_float,  on=['기준_년도', '자치구_코드_명'], how='left')
    .merge(agg_work,   on=['기준_년도', '자치구_코드_명'], how='left')
    .merge(agg_res,    on=['기준_년도', '자치구_코드_명'], how='left')
    .merge(df_관광,    on=['기준_년도', '자치구_코드_명'], how='left')
    .merge(df_임대,    on=['기준_년도', '자치구_코드_명'], how='left')
)

# ── 파생 지표 ──
자치구별_기초집계['관광매출_비중']   = 자치구별_기초집계['총_관광매출']  / 자치구별_기초집계['연간_총매출액'].replace(0, np.nan)
자치구별_기초집계['매출_유동인구비'] = 자치구별_기초집계['연간_총매출액'] / 자치구별_기초집계['총_유동인구'].replace(0, np.nan)
자치구별_기초집계['매출_직장인구비'] = 자치구별_기초집계['연간_총매출액'] / 자치구별_기초집계['총_직장인구'].replace(0, np.nan)

자치구별_기초집계 = 자치구별_기초집계.sort_values(
    ['기준_년도', '연간_총매출액'], ascending=[False, False]
).reset_index(drop=True)

print(f'자치구별_기초집계: {자치구별_기초집계.shape}')
print(f'컬럼 수: {자치구별_기초집계.shape[1]}개')
자치구별_기초집계.head()


자치구별_기초집계: (150, 29)
컬럼 수: 29개


,기준_년도,자치구_코드_명,연간_총매출액,연간_총결제건수,상권_수,평균_객단가,주중_매출비중,주말_매출비중,점심_매출비중,저녁_매출비중,...,총_직장인구,총_상주인구,내국인_관광매출,외국인_관광매출,총_관광매출,중위_월세_임대료,중위_월세_보증금,관광매출_비중,매출_유동인구비,매출_직장인구비
0,2025,강남구,11135134835002,303611013,98,36675.661811,0.781997,0.218003,0.271095,0.278029,...,4590068,707916,6.148474e+12,2.646330e+12,8.794805e+12,90.0,9073.0,0.789825,31119.887982,2.425919e+06
1,2025,송파구,8537507985607,264066386,68,32330.915399,0.729797,0.270203,0.225162,0.262521,...,2086232,992732,3.384601e+12,4.553694e+11,3.839970e+12,60.0,8500.0,0.449776,31301.288871,4.092310e+06
2,2025,중구,7851745346827,255209977,66,30765.824437,0.800111,0.199889,0.304685,0.232327,...,2541604,259728,7.583450e+12,2.746877e+12,1.033033e+13,73.0,3000.0,1.315673,34196.041440,3.089287e+06
3,2025,서초구,6060581487683,183251716,70,33072.440575,0.809003,0.190997,0.294959,0.254346,...,1864988,594812,5.322596e+12,6.752843e+11,5.997881e+12,83.0,10000.0,0.989654,25274.025101,3.249662e+06
4,2025,동대문구,5805231147773,211643817,75,27429.249907,0.735891,0.264109,0.288694,0.140065,...,211244,544424,1.014531e+12,4.544237e+10,1.059973e+12,57.0,2000.0,0.182589,24384.170878,2.748116e+07


In [72]:
# %% 상권유형별_기초집계
# ── 매출 집계 ──
agg_sales_t = sales.groupby(['기준_년도', '상권_구분_코드_명']).agg(
    연간_총매출액   = ('당월_매출_금액',        'sum'),
    연간_총결제건수 = ('당월_매출_건수',         'sum'),
    주중_매출합     = ('주중_매출_금액',          'sum'),
    주말_매출합     = ('주말_매출_금액',          'sum'),
    점심_매출합     = ('시간대_11~14_매출_금액',  'sum'),
    저녁_매출합     = ('시간대_17~21_매출_금액',  'sum'),
    상권_수         = ('상권_코드',              'nunique'),
).reset_index()

agg_sales_t['평균_객단가']   = agg_sales_t['연간_총매출액'] / agg_sales_t['연간_총결제건수'].replace(0, np.nan)
agg_sales_t['상권당_평균매출'] = agg_sales_t['연간_총매출액'] / agg_sales_t['상권_수'].replace(0, np.nan)
agg_sales_t['주중_매출비중'] = agg_sales_t['주중_매출합'] / agg_sales_t['연간_총매출액']
agg_sales_t['주말_매출비중'] = agg_sales_t['주말_매출합'] / agg_sales_t['연간_총매출액']
agg_sales_t['점심_매출비중'] = agg_sales_t['점심_매출합'] / agg_sales_t['연간_총매출액']
agg_sales_t['저녁_매출비중'] = agg_sales_t['저녁_매출합'] / agg_sales_t['연간_총매출액']
agg_sales_t = agg_sales_t.drop(columns=['주중_매출합', '주말_매출합', '점심_매출합', '저녁_매출합'])

# ── 점포 집계 ──
점포_상권분기_t = df_점포.groupby(
    ['기준_년도', '상권_구분_코드_명', '기준_년분기_코드', '상권_코드']
).agg(
    점포_수          = ('점포_수',           'sum'),
    개업_점포_수     = ('개업_점포_수',       'sum'),
    폐업_점포_수     = ('폐업_점포_수',       'sum'),
    프랜차이즈_점포_수 = ('프랜차이즈_점포_수', 'sum'),
).reset_index()

agg_store_t = 점포_상권분기_t.groupby(['기준_년도', '상권_구분_코드_명']).agg(
    총_점포수      = ('점포_수',           'mean'),
    총_개업점포수  = ('개업_점포_수',       'sum'),
    총_폐업점포수  = ('폐업_점포_수',       'sum'),
    총_프랜차이즈  = ('프랜차이즈_점포_수',  'mean'),
).reset_index()

agg_store_t['프랜차이즈_비율'] = agg_store_t['총_프랜차이즈'] / agg_store_t['총_점포수'].replace(0, np.nan)
agg_store_t['개업률']         = agg_store_t['총_개업점포수']  / agg_store_t['총_점포수'].replace(0, np.nan)
agg_store_t['폐업률']         = agg_store_t['총_폐업점포수']  / agg_store_t['총_점포수'].replace(0, np.nan)
agg_store_t['순성장률']       = agg_store_t['개업률'] - agg_store_t['폐업률']

# ── 인구 집계 (상권_구분_코드_명 원본에 있으므로 별도 머지 불필요) ──
agg_float_t = floating.groupby(['기준_년도', '상권_구분_코드_명']).agg(
    총_유동인구 = ('총_유동인구_수', 'sum')
).reset_index()

agg_work_t = df_직장.groupby(['기준_년도', '상권_구분_코드_명']).agg(
    총_직장인구 = ('총_직장_인구_수', 'sum')
).reset_index()

agg_res_t = df_상주.groupby(['기준_년도', '상권_구분_코드_명']).agg(
    총_상주인구 = ('총_상주인구_수', 'sum')
).reset_index()


# ── 전체 병합 ──
상권유형별_기초집계 = (
    agg_sales_t
    .merge(agg_store_t, on=['기준_년도', '상권_구분_코드_명'], how='left')
    .merge(agg_float_t, on=['기준_년도', '상권_구분_코드_명'], how='left')
    .merge(agg_work_t,  on=['기준_년도', '상권_구분_코드_명'], how='left')
    .merge(agg_res_t,   on=['기준_년도', '상권_구분_코드_명'], how='left')
)

상권유형별_기초집계['매출_유동인구비'] = 상권유형별_기초집계['연간_총매출액'] / 상권유형별_기초집계['총_유동인구'].replace(0, np.nan)
상권유형별_기초집계['매출_직장인구비'] = 상권유형별_기초집계['연간_총매출액'] / 상권유형별_기초집계['총_직장인구'].replace(0, np.nan)

상권유형별_기초집계 = 상권유형별_기초집계.sort_values(
    ['기준_년도', '연간_총매출액'], ascending=[False, False]
).reset_index(drop=True)

print(f'상권유형별_기초집계: {상권유형별_기초집계.shape}')
상권유형별_기초집계


상권유형별_기초집계: (24, 24)


,기준_년도,상권_구분_코드_명,연간_총매출액,연간_총결제건수,상권_수,평균_객단가,상권당_평균매출,주중_매출비중,주말_매출비중,점심_매출비중,...,총_프랜차이즈,프랜차이즈_비율,개업률,폐업률,순성장률,총_유동인구,총_직장인구,총_상주인구,매출_유동인구비,매출_직장인구비
0,2025,발달상권,56534623967697,1689715178,249,33458.079032,2.270467e+11,0.757796,0.242204,0.264573,...,NaN,NaN,NaN,NaN,NaN,1317359343,12427880,2367840,42915.112166,4.549016e+06
1,2025,골목상권,16004018089322,659221451,1039,24277.150061,1.540329e+10,0.767942,0.232058,0.243628,...,NaN,NaN,NaN,NaN,NaN,3493797117,3474872,11637620,4580.694744,4.605642e+06
2,2025,전통시장,14429375436416,519434202,283,27779.024525,5.098719e+10,0.740318,0.259682,0.266833,...,NaN,NaN,NaN,NaN,NaN,380024751,772720,760552,37969.567504,1.867349e+07
3,2025,관광특구,5690213513146,184391626,6,30859.392243,9.483689e+11,0.723461,0.276539,0.267088,...,NaN,NaN,NaN,NaN,NaN,100328058,2221588,82432,56716.073515,2.561327e+06
4,2024,발달상권,57385644139189,1778765942,249,32261.492524,2.304644e+11,0.761224,0.238776,0.263054,...,89.611446,0.095411,24.234175,28.372756,-4.138581,1334491665,11747672,2367840,43001.875279,4.884852e+06
5,2024,골목상권,16247679836134,697098682,1042,23307.575033,1.559278e+10,0.769609,0.230391,0.239966,...,13.296789,0.087721,119.672020,149.754953,-30.082933,3567449097,3383645,11637629,4554.425135,4.801828e+06
6,2024,전통시장,13524709362186,486179629,284,27818.338234,4.762222e+10,0.746215,0.253785,0.267069,...,11.643443,0.061253,26.145938,34.626271,-8.480333,395138735,769669,760706,34227.748799,1.757211e+07
7,2024,관광특구,5686081094940,183602772,6,30969.473026,9.476802e+11,0.723412,0.276588,0.268130,...,274.458333,0.050145,0.297990,0.556334,-0.258343,101396546,2201983,82432,56077.660623,2.582255e+06
8,2023,발달상권,57498800510334,1791637741,249,32092.871898,2.309189e+11,0.756882,0.243118,0.261097,...,90.320281,0.095916,32.928955,27.694602,5.234353,1335394070,10892625,2135209,43057.552675,5.278691e+06
9,2023,골목상권,16444640500411,715842157,1041,22972.439300,1.579696e+10,0.765756,0.234244,0.234231,...,13.381422,0.087594,164.957864,142.675461,22.282404,3628118861,3223555,12220136,4532.552855,5.101399e+06


-------------------------------------

# 교차테이블

In [73]:
# %% 교차테이블 1: 요일 × 상권유형 × 매출
요일_컬럼 = ['월요일_매출_금액','화요일_매출_금액','수요일_매출_금액',
            '목요일_매출_금액','금요일_매출_금액','토요일_매출_금액','일요일_매출_금액']
요일_라벨 = ['월','화','수','목','금','토','일']

cross_요일 = (
    sales.melt(id_vars=['기준_년도','상권_구분_코드_명'], value_vars=요일_컬럼,
               var_name='요일_원본', value_name='매출')
    .assign(요일=lambda x: x['요일_원본'].map(dict(zip(요일_컬럼, 요일_라벨))))
    .groupby(['기준_년도','상권_구분_코드_명','요일'])['매출'].sum()
    .reset_index()
    .pivot_table(index=['기준_년도','상권_구분_코드_명'], columns='요일', values='매출', aggfunc='sum')
    [요일_라벨].reset_index()
)
cross_요일.columns.name = None

매출합 = cross_요일[요일_라벨].sum(axis=1)
for 요일 in 요일_라벨:
    cross_요일[f'{요일}_비중'] = cross_요일[요일] / 매출합

cross_요일['주말_비중'] = (cross_요일['토'] + cross_요일['일']) / 매출합
cross_요일['피크_요일'] = cross_요일[요일_라벨].idxmax(axis=1)

print(f'요일 교차테이블: {cross_요일.shape}')
cross_요일


요일 교차테이블: (24, 18)


,기준_년도,상권_구분_코드_명,월,화,수,목,금,토,일,월_비중,화_비중,수_비중,목_비중,금_비중,토_비중,일_비중,주말_비중,피크_요일
0,2020,골목상권,1806595226355,1823391206668,1795438582243,1820772765848,1891493725798,1686149564725,1064488032651,0.151964,0.153377,0.151025,0.153156,0.159105,0.141832,0.089541,0.231373,금
1,2020,관광특구,575377493907,608989721866,611653576851,632710120790,655038847879,605556029837,415365774924,0.140176,0.148364,0.149013,0.154143,0.159583,0.147528,0.101193,0.248721,금
2,2020,발달상권,6944728702237,7038593088724,7020770166365,7101661738291,7574875738569,6996218806357,4338014125079,0.147713,0.149710,0.149331,0.151051,0.161117,0.148809,0.092269,0.241078,금
3,2020,전통시장,1305648786661,1434370811500,1377561598774,1402951399719,1472655786693,1448536740399,774483076128,0.141669,0.155636,0.149472,0.152227,0.159790,0.157173,0.084035,0.241208,금
4,2021,골목상권,2213844630215,2212219373450,2236051557727,2208626230765,2452709735793,2094271918844,1344354431550,0.149968,0.149858,0.151473,0.149615,0.166149,0.141868,0.091068,0.232936,금
5,2021,관광특구,594600803522,636436349962,648410504937,657419852090,728456441371,706332148564,484927311538,0.133421,0.142808,0.145495,0.147517,0.163456,0.158492,0.108811,0.267303,금
6,2021,발달상권,7526788389180,7723270876962,7670154261474,7760639034462,8819616151205,7939686078484,4804965980439,0.144067,0.147828,0.146811,0.148543,0.168812,0.151970,0.091970,0.243940,금
7,2021,전통시장,1663643461227,1705760913450,1744780257543,1770859583587,1939198028684,1930164302459,1062522544677,0.140785,0.144349,0.147651,0.149858,0.164103,0.163339,0.089915,0.253254,금
8,2022,골목상권,2291947855783,2382412327805,2358886265423,2370018520182,2628615230137,2285777146411,1450497284800,0.145353,0.151090,0.149598,0.150304,0.166704,0.144962,0.091989,0.236951,금
9,2022,관광특구,587282765410,628071852716,666671714613,672821332898,717669584642,692747510492,432810638468,0.133532,0.142806,0.151583,0.152981,0.163178,0.157512,0.098409,0.255921,금


In [74]:
# %% 교차테이블 2: 연령대 × 상권유형 × 매출
연령_컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
            '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령_라벨 = ['10대','20대','30대','40대','50대','60대이상']

교차테이블_매출 = (
    sales.melt(id_vars=['기준_년도','상권_구분_코드_명'], value_vars=연령_컬럼,
               var_name='연령_원본', value_name='매출')
    .assign(연령대=lambda x: x['연령_원본'].map(dict(zip(연령_컬럼, 연령_라벨))))
    .groupby(['기준_년도','상권_구분_코드_명','연령대'])['매출'].sum()
    .reset_index()
    .pivot_table(index=['기준_년도','상권_구분_코드_명'], columns='연령대', values='매출', aggfunc='sum')
    [연령_라벨].reset_index()
)
교차테이블_매출.columns.name = None

매출합 = 교차테이블_매출[연령_라벨].sum(axis=1)
for 연령 in 연령_라벨:
    교차테이블_매출[f'{연령}_비중'] = 교차테이블_매출[연령] / 매출합

교차테이블_매출['주력_연령대'] = 교차테이블_매출[연령_라벨].idxmax(axis=1)

print(f'연령대 교차테이블: {교차테이블_매출.shape}')
교차테이블_매출

연령대 교차테이블: (24, 15)


,기준_년도,상권_구분_코드_명,10대,20대,30대,40대,50대,60대이상,10대_비중,20대_비중,30대_비중,40대_비중,50대_비중,60대이상_비중,주력_연령대
0,2020,골목상권,64029490811,1478869963431,2021675542493,2692211352702,2620428084177,1973436106605,0.005901,0.136293,0.186318,0.248115,0.241500,0.181873,40대
1,2020,관광특구,20568600884,697494418139,890508534639,850009945451,658107369328,534988261073,0.005633,0.191007,0.243863,0.232772,0.180221,0.146505,30대
2,2020,발달상권,244738633634,7021893443810,9089750361453,10373154709552,8654217543008,5725506597752,0.005953,0.170810,0.221112,0.252331,0.210517,0.139275,40대
3,2020,전통시장,38693069221,841452086681,1254220496535,1647498087598,2148481239853,2610915704040,0.004530,0.098516,0.146843,0.192887,0.251541,0.305683,60대이상
4,2021,골목상권,79362200310,1682897149094,2430627682619,3299996466384,3316624439648,2686129862933,0.005881,0.124699,0.180105,0.244523,0.245755,0.199037,50대
5,2021,관광특구,21749122079,715225863441,976781864360,825288818020,713606008342,710015534649,0.005489,0.180491,0.246496,0.208266,0.180082,0.179176,30대
6,2021,발달상권,275581283806,7142662815125,9982992070690,11239475782492,9879039498106,6980067602822,0.006057,0.156982,0.219407,0.247022,0.217123,0.153409,40대
7,2021,전통시장,44574401367,916831289723,1496430814429,2001545566372,2773075334100,3744260475432,0.004061,0.083525,0.136328,0.182345,0.252632,0.341109,60대이상
8,2022,골목상권,82401754408,1730742726836,2661798179640,3398904343969,3465272274537,3024929238915,0.005737,0.120491,0.185310,0.236626,0.241246,0.210590,50대
9,2022,관광특구,21529707624,641262658015,883834798031,789865457658,703668210535,795997516882,0.005612,0.167163,0.230396,0.205900,0.183430,0.207499,30대


In [75]:
# %% 최신 연도 요약 피벗
최신_년도 = 교차테이블_매출['기준_년도'].max()
latest = 교차테이블_매출[교차테이블_매출['기준_년도'] == 최신_년도].copy()

# 비중만 추출해서 한눈에 보기
비중_컬럼 = [f'{a}_비중' for a in 연령_라벨]
요약 = latest[['상권_구분_코드_명'] + 비중_컬럼 + ['주력_연령대']].set_index('상권_구분_코드_명')
요약.columns = [c.replace('_비중', '') for c in 요약.columns]

print(f'[ {최신_년도}년 기준 상권유형별 연령대 매출 비중 ]\n')
print(요약.to_string(float_format=lambda x: f'{x:.1%}' if isinstance(x, float) else str(x)))


[ 2025년 기준 상권유형별 연령대 매출 비중 ]

            10대   20대   30대   40대   50대  60대이상 주력_연령대
상권_구분_코드_명                                           
골목상권       0.4%  8.7% 17.3% 23.0% 24.2%  26.3%  60대이상
관광특구       0.6% 13.6% 24.5% 20.3% 18.7%  22.3%    30대
발달상권       0.5% 11.4% 21.8% 23.8% 22.3%  20.2%    40대
전통시장       0.3%  5.6% 12.3% 15.8% 22.3%  43.7%  60대이상


In [76]:
# %% 교차테이블 3: 시간대 × 상권유형 × 매출
시간_컬럼 = ['시간대_00~06_매출_금액','시간대_06~11_매출_금액','시간대_11~14_매출_금액',
            '시간대_14~17_매출_금액','시간대_17~21_매출_금액','시간대_21~24_매출_금액']
시간_라벨 = ['00-06','06-11','11-14','14-17','17-21','21-24']

cross_시간 = (
    sales.melt(id_vars=['기준_년도','상권_구분_코드_명'], value_vars=시간_컬럼,
               var_name='시간_원본', value_name='매출')
    .assign(시간대=lambda x: x['시간_원본'].map(dict(zip(시간_컬럼, 시간_라벨))))
    .groupby(['기준_년도','상권_구분_코드_명','시간대'])['매출'].sum()
    .reset_index()
    .pivot_table(index=['기준_년도','상권_구분_코드_명'], columns='시간대', values='매출', aggfunc='sum')
    [시간_라벨].reset_index()
)
cross_시간.columns.name = None

매출합 = cross_시간[시간_라벨].sum(axis=1)
for 시간 in 시간_라벨:
    cross_시간[f'{시간}_비중'] = cross_시간[시간] / 매출합

cross_시간['피크_시간대'] = cross_시간[시간_라벨].idxmax(axis=1)

print(f'시간대 교차테이블: {cross_시간.shape}')
cross_시간

시간대 교차테이블: (24, 15)


,기준_년도,상권_구분_코드_명,00-06,06-11,11-14,14-17,17-21,21-24,00-06_비중,06-11_비중,11-14_비중,14-17_비중,17-21_비중,21-24_비중,피크_시간대
0,2020,골목상권,414396511976,1301449433773,2695150914906,2795443698386,3527244097049,1154644448198,0.034857,0.109473,0.226706,0.235142,0.296698,0.097124,17-21
1,2020,관광특구,97945818033,336377425526,1064361212665,1165266536357,1140620512046,300120061427,0.023862,0.081950,0.259304,0.283887,0.277882,0.073116,14-17
2,2020,발달상권,1399580338251,4648930563164,11896868413292,13080962734555,12929363462995,3059156853365,0.029769,0.098882,0.253045,0.278230,0.275006,0.065068,14-17
3,2020,전통시장,226225796243,1351461536560,2325518086516,2606133263932,2193251345013,513618171610,0.024547,0.146640,0.252329,0.282777,0.237978,0.055730,14-17
4,2021,골목상권,237616636741,1655007130709,3518287218239,3481238012468,4554736195964,1315192684223,0.016096,0.112112,0.238333,0.235823,0.308543,0.089093,17-21
5,2021,관광특구,48826838223,349595728287,1245073570173,1308996220270,1235185474320,268905580711,0.010956,0.078445,0.279378,0.293722,0.277160,0.060339,14-17
6,2021,발달상권,956851753438,5483049758862,13927943411414,14335230975800,14369565088653,3172479784039,0.018315,0.104949,0.266588,0.274384,0.275041,0.060723,17-21
7,2021,전통시장,147726360991,1783576192367,3156910377933,3419382816879,2771973714709,537359628748,0.012501,0.150934,0.267152,0.289363,0.234576,0.045474,14-17
8,2022,골목상권,461984517508,1833689768170,3668142801060,3489271414461,4755651073231,1559415056111,0.029299,0.116291,0.232630,0.221286,0.301598,0.098896,17-21
9,2022,관광특구,123214854613,361237120710,1211463180495,1180370061967,1147524622614,374265558840,0.028016,0.082135,0.275453,0.268383,0.260915,0.085098,11-14


In [77]:
# %% 교차테이블 4: 성별 × 자치구 × 매출
성별_컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별_라벨 = ['남성', '여성']

cross_성별 = (
    sales.melt(id_vars=['기준_년도','자치구_코드_명'], value_vars=성별_컬럼,
               var_name='성별_원본', value_name='매출')
    .assign(성별=lambda x: x['성별_원본'].map(dict(zip(성별_컬럼, 성별_라벨))))
    .groupby(['기준_년도','자치구_코드_명','성별'])['매출'].sum()
    .reset_index()
    .pivot_table(index=['기준_년도','자치구_코드_명'], columns='성별', values='매출', aggfunc='sum')
    [성별_라벨].reset_index()
)
cross_성별.columns.name = None

매출합 = cross_성별[성별_라벨].sum(axis=1)
cross_성별['남성_비중'] = cross_성별['남성'] / 매출합
cross_성별['여성_비중'] = cross_성별['여성'] / 매출합
cross_성별['주력_성별'] = cross_성별[성별_라벨].idxmax(axis=1)

print(f'성별 교차테이블: {cross_성별.shape}')
cross_성별

성별 교차테이블: (150, 7)


,기준_년도,자치구_코드_명,남성,여성,남성_비중,여성_비중,주력_성별
0,2020,강남구,4103935986383,4842098936056,0.458744,0.541256,여성
1,2020,강동구,922380116181,886508860457,0.509915,0.490085,남성
2,2020,강북구,504241283817,471490259193,0.516783,0.483217,남성
3,2020,강서구,1054935146889,943448459592,0.527894,0.472106,남성
4,2020,관악구,762473873288,733241528843,0.509772,0.490228,남성
...,...,...,...,...,...,...,...
145,2025,용산구,2026481543102,1479512896344,0.578005,0.421995,남성
146,2025,은평구,654289274575,712249278842,0.478793,0.521207,여성
147,2025,종로구,2359698888807,1970927712060,0.544886,0.455114,남성
148,2025,중구,3039297378609,3432209777809,0.469643,0.530357,여성


In [78]:
# %% 교차테이블 5: 업종 × 자치구 × 매출
# 전체 기간 합산 기준 상위 20개 업종만 사용 (너무 많으면 테이블이 넓어짐)
top_업종 = (
    sales.groupby('서비스_업종_코드_명')['당월_매출_금액'].sum()
    .nlargest(20).index.tolist()
)

cross_업종 = (
    sales[sales['서비스_업종_코드_명'].isin(top_업종)]
    .groupby(['기준_년도','자치구_코드_명','서비스_업종_코드_명'])['당월_매출_금액'].sum()
    .reset_index()
    .pivot_table(index=['기준_년도','자치구_코드_명'], columns='서비스_업종_코드_명',
                 values='당월_매출_금액', aggfunc='sum')
    .reset_index()
)
cross_업종.columns.name = None

# 각 자치구의 주력 업종
업종_컬럼 = [c for c in cross_업종.columns if c not in ['기준_년도','자치구_코드_명']]
cross_업종['주력_업종'] = cross_업종[업종_컬럼].idxmax(axis=1)

print(f'업종 교차테이블: {cross_업종.shape}  (업종 수: {len(업종_컬럼)}개)')
cross_업종

업종 교차테이블: (150, 23)  (업종 수: 20개)


,기준_년도,자치구_코드_명,문구,반찬가게,수산물판매,슈퍼마켓,시계및귀금속,양식음식점,육류판매,의약품,...,조명용품,청과상,치과의원,커피-음료,컴퓨터및주변장치판매,편의점,한식음식점,호프-간이주점,화장품,주력_업종
0,2020,강남구,1.212560e+11,8.406591e+10,2.835845e+09,2.347271e+11,1.156105e+11,2.278359e+11,3.515659e+10,5.180131e+11,...,2.632238e+10,2.140651e+10,1.840804e+11,2.978478e+11,6.511366e+10,4.411858e+11,1.102857e+12,1.240230e+11,1.286721e+12,일반의류
1,2020,강동구,7.561418e+09,3.431078e+10,2.231904e+10,7.082864e+10,4.458222e+09,2.343083e+10,6.273685e+10,2.021003e+11,...,1.040125e+10,3.485586e+10,7.569488e+10,2.447139e+10,4.835062e+09,8.710797e+10,3.014215e+11,4.514815e+10,6.450513e+10,한식음식점
2,2020,강북구,1.179706e+08,7.430215e+09,1.566799e+10,5.733494e+10,2.106082e+09,5.769536e+09,4.194601e+10,1.011075e+11,...,6.662938e+09,3.101536e+10,4.836306e+10,1.315628e+10,6.370211e+07,6.116587e+10,2.015025e+11,3.008234e+10,3.580414e+10,한식음식점
3,2020,강서구,3.016170e+10,1.281215e+11,3.337773e+11,1.011379e+11,1.171772e+09,5.627389e+09,1.190207e+11,1.511232e+11,...,2.933379e+09,2.438209e+10,6.133769e+10,2.008648e+10,1.805572e+10,1.497429e+11,3.201542e+11,4.519948e+10,1.156887e+11,수산물판매
4,2020,관악구,3.731953e+09,1.556898e+10,1.444712e+10,1.044947e+11,2.215547e+09,1.408772e+10,8.020962e+10,7.634126e+10,...,2.906567e+09,3.595121e+10,7.130850e+10,3.205935e+10,9.728655e+09,1.167933e+11,2.491139e+11,4.338758e+10,4.399579e+10,한식음식점
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2025,용산구,2.928706e+10,3.253396e+10,4.608250e+09,1.144239e+11,8.348101e+09,2.272999e+11,3.188062e+10,6.155108e+10,...,4.002258e+09,2.639482e+10,2.496364e+10,9.967509e+10,1.831236e+12,1.008157e+11,5.723227e+11,2.026228e+11,2.840180e+10,컴퓨터및주변장치판매
146,2025,은평구,1.720943e+10,7.771961e+10,8.662153e+09,1.181120e+11,2.830232e+09,3.980174e+09,3.633405e+10,1.283373e+11,...,2.915147e+09,1.945452e+10,6.912707e+10,2.498232e+10,9.801928e+07,8.145192e+10,3.025002e+11,6.200038e+10,6.349903e+09,한식음식점
147,2025,종로구,2.387471e+11,1.571603e+10,4.325208e+09,1.411346e+11,8.153932e+11,1.404956e+11,2.696054e+10,3.344915e+11,...,3.068103e+11,2.180135e+10,6.017404e+10,2.803063e+11,9.756065e+09,1.349567e+11,1.114127e+12,2.609392e+11,3.009898e+10,한식음식점
148,2025,중구,1.902841e+11,2.712586e+11,1.841801e+10,2.395752e+11,2.152007e+11,1.143499e+11,2.369937e+11,2.044527e+11,...,3.542854e+11,3.095828e+11,1.423109e+11,3.654080e+11,9.384614e+09,2.979905e+11,1.602645e+12,1.963620e+11,2.659995e+11,한식음식점


----------------------------------------------------

# RFM로직용 테이블 생성

In [79]:
# %% 상권별_RFM기초 테이블 생성
# ── 2025년에 포함된 분기만 추출하여 동일 분기 기준 비교 (부분 연도 왜곡 방지) ──
분기_2025 = sorted(sales[sales['기준_년도'] == 2025]['기준_년분기_코드'].unique())
분기_2024_동일 = [q - 10 for q in 분기_2025]

s_2025 = sales[sales['기준_년분기_코드'].isin(분기_2025)]
s_2024 = sales[sales['기준_년분기_코드'].isin(분기_2024_동일)]

print(f'비교 기준 분기 수: {len(분기_2025)}개')
print(f'  2025 분기: {분기_2025}')
print(f'  2024 동일 분기: {분기_2024_동일}')

# ── 상권별 집계 ──
def agg_by_상권(df, suffix):
    return df.groupby('상권_코드').agg(**{
        f'매출_{suffix}':    ('당월_매출_금액', 'sum'),
        f'결제건수_{suffix}': ('당월_매출_건수', 'sum'),
    })

rfm_base = agg_by_상권(s_2024, '2024').join(agg_by_상권(s_2025, '2025'), how='inner')

# ── 유동인구 (2024~2025 평균) ──
float_avg = (
    floating[floating['기준_년도'].isin([2024, 2025])]
    .groupby('상권_코드')['총_유동인구_수']
    .mean()
    .rename('평균_유동인구')
)

# ── 상권 메타 (좌표·자치구·상권유형) ──
상권_meta = (
    area[['상권_코드','상권_코드_명','자치구_코드_명','상권_구분_코드_명','엑스좌표_값','와이좌표_값']]
    .drop_duplicates('상권_코드')
    .set_index('상권_코드')
)

rfm = rfm_base.join(float_avg, how='left').join(상권_meta, how='left')

# ── R: 2024→2025 매출 증감률 ──
rfm['R_매출증감률']    = (rfm['매출_2025'] - rfm['매출_2024']) / rfm['매출_2024'].replace(0, np.nan)
# ── R 보조: 결제건수 증감률 (인플레이션 필터링용) ──
rfm['결제건수_증감률'] = (rfm['결제건수_2025'] - rfm['결제건수_2024']) / rfm['결제건수_2024'].replace(0, np.nan)

# ── F: 최근 2년 평균 결제건수 ──
rfm['F_결제건수'] = (rfm['결제건수_2024'] + rfm['결제건수_2025']) / 2

# ── M: 2025 기준 객단가 ──
rfm['M_객단가'] = rfm['매출_2025'] / rfm['결제건수_2025'].replace(0, np.nan)

# ── 결제전환율 (디커플링 EDA용) ──
rfm['결제전환율'] = rfm['F_결제건수'] / rfm['평균_유동인구'].replace(0, np.nan)

# ── 진짜성장 플래그: 매출·결제건수 모두 증가 ──
rfm['진짜성장_플래그'] = (rfm['R_매출증감률'] > 0) & (rfm['결제건수_증감률'] > 0)

rfm = rfm.reset_index().sort_values('R_매출증감률', ascending=False).reset_index(drop=True)

print(f'\n상권별_RFM기초: {rfm.shape}')
print(f'진짜성장 상권: {rfm["진짜성장_플래그"].sum()}개 / 전체 {len(rfm)}개')
rfm.head(10)

비교 기준 분기 수: 4개
  2025 분기: [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]
  2024 동일 분기: [np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244)]

상권별_RFM기초: (1575, 17)
진짜성장 상권: 373개 / 전체 1575개


,상권_코드,매출_2024,결제건수_2024,매출_2025,결제건수_2025,평균_유동인구,상권_코드_명,자치구_코드_명,상권_구분_코드_명,엑스좌표_값,와이좌표_값,R_매출증감률,결제건수_증감률,F_결제건수,M_객단가,결제전환율,진짜성장_플래그
0,3110238,2844923,71,932345235,1421,109625.000,배꽃어린이공원,중랑구,골목상권,206668,456017,326.722485,19.014085,746.0,656119.095707,0.006805,True
1,3130305,100244621,4506,2228266257,93458,29802.375,논현종합시장,강남구,전통시장,203038,445603,21.228288,19.740790,48982.0,23842.434644,1.643560,True
2,3130046,4476390498,225453,59128312308,2029651,66316.625,team204(팀204),중구,전통시장,201168,452028,12.208926,8.002546,1127552.0,29132.255894,17.002554,True
3,3110773,164716594,55,1843989966,434784,87162.375,당산역 13번,영등포구,골목상권,191087,448516,10.194925,7904.163636,217419.5,4241.163350,2.494419,True
4,3110563,44875963,474,358632127,6644,174187.625,상수역 3번,마포구,골목상권,193303,449582,6.991631,13.016878,3559.0,53978.345424,0.020432,True
5,3110945,9857950,33,77417765,165,24653.500,본마을노인복지센터(서초구립내곡도서관),서초구,골목상권,204653,439500,6.853333,4.000000,99.0,469198.575758,0.004016,True
6,3130243,48039960,1135,314989029,3249,53471.875,도림시장,영등포구,전통시장,191026,445598,5.556813,1.862555,2192.0,96949.531856,0.040994,True
7,3110281,259923609,229,1661779793,9368,68530.875,길상사,성북구,골목상권,199476,455329,5.393339,39.908297,4798.5,177388.961678,0.070020,True
8,3110546,1792683519,90230,7282248459,665029,722819.125,양화대교북단,마포구,골목상권,191662,450075,3.062205,6.370376,377629.5,10950.272032,0.522440,True
9,3110764,1184937003,30144,4800378106,148207,67212.125,문래역 3번,영등포구,골목상권,190693,446831,3.051167,3.916633,89175.5,32389.685413,1.326777,True


In [80]:
# %% CSV 저장
OUTPUT_DIR = project_root / 'data' / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

tables = {
    '자치구별_기초집계.csv':    자치구별_기초집계,
    '상권유형별_기초집계.csv':  상권유형별_기초집계,
    'cross_요일.csv':          cross_요일,
    'cross_연령대.csv':        교차테이블_매출,
    'cross_시간대.csv':        cross_시간,
    'cross_성별.csv':          cross_성별,
    'cross_업종.csv':          cross_업종,
    '상권별_RFM기초.csv':      rfm,
}

for fname, df in tables.items():
    df.to_csv(OUTPUT_DIR / fname, index=False, encoding='utf-8-sig')

print('✅ CSV 저장 완료')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  {f.name:40s}  {f.stat().st_size / 1024:7.1f} KB')

✅ CSV 저장 완료
  cross_성별.csv                                 13.1 KB
  cross_시간대.csv                                 5.5 KB
  cross_업종.csv                                 46.5 KB
  cross_연령대.csv                                 5.5 KB
  cross_요일.csv                                  6.7 KB
  상권별_RFM기초.csv                               322.3 KB
  상권유형별_기초집계.csv                                8.2 KB
  자치구별_기초집계.csv                                56.2 KB
